In [1]:
!git clone https://github.com/dgvoelkel/Voelkel-Informatics

Cloning into 'Voelkel-Informatics'...


In [2]:
import pandas as pd
arm_angles_df = pd.read_csv("/content/Voelkel-Informatics/fangraphs-leaderboards-34.csv")
arm_angles_df

FileNotFoundError: [Errno 2] No such file or directory: '/content/Voelkel-Informatics/fangraphs-leaderboards-34.csv'

## We have some missing data!
Therefore, lets see which pitches are overwhelmingly missing from our dataset, and drop those pitches.

In [ ]:
# Columns with at least one NaN, along with dtype and NaN count
nan_dtypes = (
    arm_angles_df
    .isna()
    .sum()
    .reset_index(name="NaN_Count")
    .rename(columns={"index": "Column"})
)

nan_dtypes["Dtype"] = nan_dtypes["Column"].map(arm_angles_df.dtypes)

nan_dtypes = nan_dtypes[nan_dtypes["NaN_Count"] > 0]

nan_dtypes

,Column,NaN_Count,Dtype
5,vFA (pi),6,float64
7,FA-AA,7,float64
8,FT-AA,191,float64
9,FC-AA,73,float64
10,FS-AA,146,float64
11,FO-AA,189,float64
12,SI-AA,26,float64
13,SL-AA,15,float64
14,CU-AA,57,float64
15,KC-AA,155,float64


Based on the NaN counts, we will drop columns with more than 300 missing values. Therefore, when we drop all rows with NaN values, we will still have a sizeable dataset with enough pitchers.

Our remaining features are fastballs, sinkers, slider, change-ups, and off-speed pitches

In [ ]:
arm_angles_df = arm_angles_df[["NameASCII","FA-AA","SI-AA","SL-AA","CH-AA","SLO-AA","IP","PlayerId"]].dropna()
arm_angles_df.set_index("PlayerId", inplace=True)
arm_angles_df


,NameASCII,FA-AA,SI-AA,SL-AA,CH-AA,SLO-AA,IP
PlayerId,,,,,,,
13164,Eduardo Rodriguez,39.415129,41.253285,42.476957,39.420162,42.476957,50.0
14120,Lance McCullers Jr.,29.039615,24.937280,21.997316,25.219089,21.997316,51.1
14361,Ty Blach,20.934163,20.765656,22.499424,19.879493,22.499424,56.1
16977,Trevor Williams,17.790392,15.834486,16.697712,15.851582,17.099269,66.2
19666,Jason Alexander,27.710999,19.322953,20.377431,22.845787,17.882393,68.1
...,...,...,...,...,...,...,...
19291,Zac Gallen,44.695675,45.815363,41.676149,35.612502,41.676149,192.0
13743,Max Fried,50.777536,44.703614,40.807485,39.919152,43.394454,195.1
16137,Carlos Rodon,47.094003,43.013839,40.456103,43.414025,40.456103,195.1


In [ ]:
cols = ["FA-AA","SI-AA","SL-AA","CH-AA","SLO-AA","IP"]

corr_with_ip = arm_angles_df[cols].corr(numeric_only=True)["IP"].sort_values(ascending=False).round(4)

display(corr_with_ip)

,IP
IP,1.0000
FA-AA,0.0763
CH-AA,0.0663
SI-AA,0.0570
SLO-AA,0.0391
SL-AA,0.0175


## Regression Model Benchmark — All sklearn Regressors

Every sklearn regressor is evaluated with **5-fold cross-validation** (`cross_validate`) on:
- **Val MAE** — mean absolute error on held-out folds (lower is better)
- **Val R²** — coefficient of determination on held-out folds (higher is better)
- **Latency (s)** — average fit + score time per fold

All models are wrapped in a `StandardScaler` pipeline. Results are sorted by Val MAE ascending.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import cross_validate, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    BayesianRidge, ARDRegression, HuberRegressor,
    Lars, LassoLars, OrthogonalMatchingPursuit,
    TweedieRegressor, PoissonRegressor, GammaRegressor,
    SGDRegressor, QuantileRegressor, TheilSenRegressor,
    RANSACRegressor, PassiveAggressiveRegressor,
)
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    GradientBoostingRegressor, AdaBoostRegressor,
    BaggingRegressor, HistGradientBoostingRegressor,
)
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.svm import SVR, NuSVR, LinearSVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_ridge import KernelRidge

In [ ]:
FEATURES = ["FA-AA", "SI-AA", "SL-AA", "CH-AA", "SLO-AA"]
X = arm_angles_df[FEATURES].values
y = arm_angles_df["IP"].values

models = {
    "DummyRegressor":              DummyRegressor(strategy="mean"),
    "LinearRegression":            LinearRegression(),
    "Ridge":                       Ridge(),
    "Lasso":                       Lasso(),
    "ElasticNet":                  ElasticNet(),
    "BayesianRidge":               BayesianRidge(),
    "ARDRegression":               ARDRegression(),
    "HuberRegressor":              HuberRegressor(),
    "Lars":                        Lars(),
    "LassoLars":                   LassoLars(),
    "OrthogonalMatchingPursuit":   OrthogonalMatchingPursuit(),
    "TweedieRegressor":            TweedieRegressor(),
    "PoissonRegressor":            PoissonRegressor(),
    "GammaRegressor":              GammaRegressor(),
    "SGDRegressor":                SGDRegressor(max_iter=1000, random_state=42),
    "QuantileRegressor":           QuantileRegressor(),
    "TheilSenRegressor":           TheilSenRegressor(random_state=42),
    "RANSACRegressor":             RANSACRegressor(random_state=42),
    "PassiveAggressiveRegressor":  PassiveAggressiveRegressor(max_iter=1000, random_state=42),
    "DecisionTreeRegressor":       DecisionTreeRegressor(random_state=42),
    "ExtraTreeRegressor":          ExtraTreeRegressor(random_state=42),
    "RandomForestRegressor":       RandomForestRegressor(n_estimators=100, random_state=42),
    "ExtraTreesRegressor":         ExtraTreesRegressor(n_estimators=100, random_state=42),
    "GradientBoostingRegressor":   GradientBoostingRegressor(random_state=42),
    "AdaBoostRegressor":           AdaBoostRegressor(random_state=42),
    "BaggingRegressor":            BaggingRegressor(random_state=42),
    "HistGradientBoosting":        HistGradientBoostingRegressor(random_state=42),
    "SVR":                         SVR(),
    "NuSVR":                       NuSVR(),
    "LinearSVR":                   LinearSVR(max_iter=2000),
    "KNeighborsRegressor":         KNeighborsRegressor(),
    "GaussianProcessRegressor":    GaussianProcessRegressor(),
    "MLPRegressor":                MLPRegressor(max_iter=1000, random_state=42),
    "PLSRegression":               PLSRegression(n_components=2),
    "KernelRidge":                 KernelRidge(),
}

DEFAULT_K = 5
cv = KFold(n_splits=DEFAULT_K, shuffle=True, random_state=42)
scoring = ["neg_mean_absolute_error", "r2"]

rows = []
for name, model in models.items():
    pipe = make_pipeline(StandardScaler(), model)
    try:
        res = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
        rows.append({
            "Model":       name,
            "Val MAE":     round(-res["test_neg_mean_absolute_error"].mean(), 4),
            "Val R²":      round(res["test_r2"].mean(), 4),
            "Latency (s)": round(res["fit_time"].mean() + res["score_time"].mean(), 5),
        })
    except Exception as e:
        rows.append({"Model": name, "Val MAE": None, "Val R²": None, "Latency (s)": None})

model_results_df = (
    pd.DataFrame(rows)
    .sort_values("Val MAE", na_position="last")
    .reset_index(drop=True)
)
display(model_results_df)

## K-Fold Optimization on Best Model

Identify the model with the lowest Val MAE, then sweep `k` from 2 to 30 to find the optimal number of folds for both MAE and R².

In [ ]:
best_name = model_results_df.dropna(subset=["Val MAE"]).iloc[0]["Model"]
best_estimator = make_pipeline(StandardScaler(), models[best_name])
print(f"Best model: {best_name}\n")

k_rows = []
for k in range(2, 31):
    cv_k = KFold(n_splits=k, shuffle=True, random_state=42)
    try:
        res = cross_validate(best_estimator, X, y, cv=cv_k, scoring=scoring)
        k_rows.append({
            "k":       k,
            "Val MAE": round(-res["test_neg_mean_absolute_error"].mean(), 4),
            "Val R²":  round(res["test_r2"].mean(), 4),
        })
    except Exception as e:
        k_rows.append({"k": k, "Val MAE": None, "Val R²": None})

k_results_df = pd.DataFrame(k_rows)
display(k_results_df)

best_k_mae = k_results_df.loc[k_results_df["Val MAE"].idxmin(), "k"]
best_k_r2  = k_results_df.loc[k_results_df["Val R²"].idxmax(), "k"]
print(f"\nOptimal k by Val MAE : {best_k_mae}")
print(f"Optimal k by Val R²  : {best_k_r2}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict

dummy_pipe = make_pipeline(StandardScaler(), DummyRegressor(strategy="mean"))
cv4 = KFold(n_splits=4, shuffle=True, random_state=42)

y_pred = cross_val_predict(dummy_pipe, X, y, cv=cv4)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y, y_pred, alpha=0.7, edgecolors="k", linewidths=0.4, color="steelblue", s=60)

lims = [min(y.min(), y_pred.min()) - 5, max(y.max(), y_pred.max()) + 5]
ax.plot(lims, lims, "r--", linewidth=1.2, label="Perfect prediction")

ax.set_xlabel("Actual IP", fontsize=12)
ax.set_ylabel("Predicted IP", fontsize=12)
ax.set_title("DummyRegressor (mean) — Predicted vs. Actual IP\n4-Fold Cross-Validation", fontsize=13)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()